# Dataflow Inventory

This notebook builds the Dataflow inventory delta tables (`ScanResults`, `Dataflows`, `Datasets`) from a **pre-existing Power BI Scanner API JSON** stored in the lakehouse. The original tenant-scan logic (calls to `admin/workspaces/getInfo`) has been **disabled** — the input is the customer-provided `Scanner_API.json`.

**Input file:** `Files/Scanner_API/Scanner_API.json` in lakehouse `DFG2_Migration`.
**Expected JSON shape:** `{ "workspaces": [ ... ] }` (same shape produced by the Scanner API `getInfo` endpoint).

## Load Scanner API JSON from Lakehouse

Reads the customer-provided Scanner API export and exposes it as `scan_results` for the downstream cells. Streaming parser used to handle large files (multi-GB) with low memory overhead.

In [3]:
import json
import os

SCANNER_JSON_PATH = "/lakehouse/default/Files/Scanner_API/Scanner_API.json"

if not os.path.exists(SCANNER_JSON_PATH):
    raise FileNotFoundError(
        f"Scanner_API.json not found at {SCANNER_JSON_PATH}. "
        "Upload the customer-provided file to the lakehouse Files/Scanner_API/ folder."
    )

size_mb = os.path.getsize(SCANNER_JSON_PATH) / (1024 * 1024)
print(f"Loading {SCANNER_JSON_PATH} ({size_mb:,.1f} MB)...")

with open(SCANNER_JSON_PATH, "r", encoding="utf-8") as f:
    scan_results = json.load(f)

# Normalize top-level shape: the rest of the notebook expects {'workspaces': [...]}
if isinstance(scan_results, list):
    scan_results = {"workspaces": scan_results}
elif isinstance(scan_results, dict) and "workspaces" not in scan_results:
    # some Scanner API exports wrap the payload under 'value'
    if "value" in scan_results and isinstance(scan_results["value"], list):
        scan_results = {"workspaces": scan_results["value"]}
    else:
        raise ValueError(
            "Unexpected JSON shape: top-level dict has no 'workspaces' or 'value' key."
        )

print(f"Loaded {len(scan_results['workspaces'])} workspaces from Scanner_API.json")

StatementMeta(, 6454c314-ecbe-4717-9f33-89c8ff242476, 5, Finished, Available, Finished, False)

Loading /lakehouse/default/Files/Scanner_API/Scanner_API.json (2,174.5 MB)...
Loaded 4714 workspaces from Scanner_API.json


## Build delta tables for reporting

In [ ]:
import shutil
import pandas as pd
import json, os, shutil

BATCH_SIZE = 50  # only affects pandas chunking, not Spark RPC

STAGING_BASE = "/lakehouse/default/Files/_staging_inventory"
for sub in ("ScanResults", "Dataflows", "Datasets"):
    p = f"{STAGING_BASE}/{sub}"
    if os.path.exists(p):
        shutil.rmtree(p)
    os.makedirs(p, exist_ok=True)

# Original ScanResults columns (kept intact so the semantic model doesn't break).
# Heavy nested arrays (reports, dashboards, datasets, dataflows, datamarts, warehouses, users)
# are JSON-stringified before parquet write — that's safe now because we no longer use
# spark.createDataFrame per batch, so the RPC closure is not affected.
ws_base_cols = [
    "id", "name", "state", "isOnDedicatedCapacity", "capacityId",
    "defaultDatasetStorageFormat", "domainId", "Lakehouse", "Reflex",
    "reports", "dashboards", "datasets", "dataflows", "datamarts",
    "warehouses", "SQLAnalyticsEndpoint", "users",
]

workspaces_all = scan_results["workspaces"]
total = len(workspaces_all)
print(f"Total workspaces: {total}. Building inventory in batches of {BATCH_SIZE}.")

# ---- discover all fields present in source so the final tables include every
# possible column the semantic model may reference (as null when absent in data).
print("Discovering all dataset/dataflow fields in source...")
all_ds_fields = set()
all_df_fields = set()
for ws in workspaces_all:
    if not isinstance(ws, dict):
        continue
    for ds in ws.get("datasets") or []:
        if isinstance(ds, dict):
            all_ds_fields.update(ds.keys())
    for dfx in ws.get("dataflows") or []:
        if isinstance(dfx, dict):
            all_df_fields.update(dfx.keys())
expected_ds_cols = [f"datasets.{k}" for k in sorted(all_ds_fields)]
expected_df_cols = [f"dataflows.{k}" for k in sorted(all_df_fields)]
# Force-include columns referenced by the semantic model that the Scanner API
# may omit when no entity in the current scan exposes them (e.g. tags).
# Without this, the model refresh fails with "column not found in delta table".
MODEL_REQUIRED_DS_COLS = ["datasets.tags"]
MODEL_REQUIRED_DF_COLS = ["dataflows.tags"]
for c in MODEL_REQUIRED_DS_COLS:
    if c not in expected_ds_cols:
        expected_ds_cols.append(c)
for c in MODEL_REQUIRED_DF_COLS:
    if c not in expected_df_cols:
        expected_df_cols.append(c)
print(f"  found {len(expected_ds_cols)} dataset fields, {len(expected_df_cols)} dataflow fields")


def _json_safe(v):
    if v is None or isinstance(v, (str, int, float, bool)):
        return v
    try:
        return json.dumps(v, default=str, ensure_ascii=False)
    except Exception:
        return str(v)


def normalize_workspaces(batch):
    df = pd.json_normalize(batch, errors="ignore")
    df = df.reindex(columns=ws_base_cols)
    return df.rename(columns={"id": "WorkspaceID", "name": "WorkspaceName"})


def normalize_dataflows(batch):
    df = pd.json_normalize(
        batch,
        record_path=["dataflows"],
        meta=["id", "name", "isOnDedicatedCapacity", "capacityId"],
        record_prefix="dataflows.",
        errors="ignore",
    )
    return df.rename(columns={
        "id": "WorkspaceID",
        "name": "WorkspaceName",
        "dataflows.objectId": "DataflowID",
        "dataflows.name": "DataFlowName",
        "dataflows.modifiedBy": "DataflowModifiedBy",
        "dataflows.modifiedDateTime": "DataflowModifiedDate",
        "dataflows.relations": "Relations",
        "dataflows.refreshSchedule.enabled": "RefreshEnabled",
        "dataflows.generation": "DataflowGeneration",
        "dataflows.description": "DataflowDescription",
        "dataflows.datasourceUsages": "DatasourceUsages",
    })


def normalize_datasets(batch):
    df = pd.json_normalize(
        batch,
        record_path=["datasets"],
        meta=["id", "name", "isOnDedicatedCapacity", "capacityId"],
        record_prefix="datasets.",
        errors="ignore",
    )
    df = df.rename(columns={
        "id": "WorkspaceID",
        "name": "WorkspaceName",
        "datasets.id": "DatasetID",
        "datasets.name": "DatasetName",
        "datasets.createdDate": "DatasetCreatedDate",
        "datasets.upstreamDataflows": "UpstreamDataflows",
    })
    if "UpstreamDataflows" in df.columns:
        df = df[df["UpstreamDataflows"].notna()]
        df = df.explode("UpstreamDataflows", ignore_index=True)
        df["DataflowID"] = df["UpstreamDataflows"].apply(
            lambda x: x.get("targetDataflowId") if isinstance(x, dict) else None
        )
        df["DataflowWorkspaceID"] = df["UpstreamDataflows"].apply(
            lambda x: x.get("groupId") if isinstance(x, dict) else None
        )
        df.drop(columns=["UpstreamDataflows"], inplace=True)
    else:
        df = df.iloc[0:0]
    return df


ws_chunks, df_chunks, ds_chunks = [], [], []
n_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
for i, start in enumerate(range(0, total, BATCH_SIZE), start=1):
    batch = workspaces_all[start:start + BATCH_SIZE]
    ws_chunks.append(normalize_workspaces(batch))
    df_chunks.append(normalize_dataflows(batch))
    ds_chunks.append(normalize_datasets(batch))
    if i % 20 == 0 or i == n_batches:
        print(f"  processed batch {i}/{n_batches}")

print("Concatenating per-batch DataFrames (single pandas DF per table)...")
ws_df = pd.concat(ws_chunks, ignore_index=True) if ws_chunks else pd.DataFrame()
dfl_df = pd.concat(df_chunks, ignore_index=True) if df_chunks else pd.DataFrame()
ds_df = pd.concat(ds_chunks, ignore_index=True) if ds_chunks else pd.DataFrame()

# Ensure every expected column exists (as null) so semantic model never breaks
# on a column-not-found error after a rebuild.
for col in expected_df_cols:
    # skip the renamed ones — they keep the new name in dfl_df
    renamed = {
        "dataflows.objectId", "dataflows.name", "dataflows.modifiedBy",
        "dataflows.modifiedDateTime", "dataflows.relations",
        "dataflows.refreshSchedule.enabled", "dataflows.generation",
        "dataflows.description", "dataflows.datasourceUsages",
    }
    if col in renamed:
        continue
    if col not in dfl_df.columns:
        dfl_df[col] = pd.NA

for col in expected_ds_cols:
    renamed = {
        "datasets.id", "datasets.name", "datasets.createdDate",
        "datasets.upstreamDataflows",
    }
    if col in renamed:
        continue
    if col not in ds_df.columns:
        ds_df[col] = pd.NA

print(f"  ScanResults: {len(ws_df)} rows, {len(ws_df.columns)} cols")
print(f"  Dataflows:   {len(dfl_df)} rows, {len(dfl_df.columns)} cols")
print(f"  Datasets:    {len(ds_df)} rows, {len(ds_df.columns)} cols")


import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql.types import StructType, StructField, StringType


def write_table(pdf, table):
    if pdf is None or pdf.empty:
        print(f"  {table}: no rows, skipping")
        return
    # Stringify nested objects, then force every column to plain Python str / None.
    # We then write parquet via pyarrow with an EXPLICIT all-string schema so the
    # parquet file metadata unambiguously declares STRING (no boolean/int leftovers
    # from pandas dtype inference).
    for col in pdf.columns:
        if pdf[col].dtype == object:
            pdf[col] = pdf[col].apply(_json_safe)
        pdf[col] = pdf[col].map(lambda v: None if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    pa_schema = pa.schema([(c, pa.string()) for c in pdf.columns])
    arrays = [pa.array(pdf[c].tolist(), type=pa.string()) for c in pdf.columns]
    pa_table = pa.Table.from_arrays(arrays, schema=pa_schema)
    path = f"{STAGING_BASE}/{table}/data.parquet"
    pq.write_table(pa_table, path)
    # Drop existing table + physically remove its delta files so Spark cannot
    # pick up the old schema from a stale _delta_log.
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    shutil.rmtree(f"/lakehouse/default/Tables/{table}", ignore_errors=True)
    # Force Spark to use an explicit all-StringType schema when reading the parquet
    # (defeats any cached schema inference / lakehouse catalog stale state).
    spark_schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    sdf = spark.read.schema(spark_schema).parquet(f"file://{path}")
    (sdf.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(table))
    print(f"  {table}: wrote {sdf.count()} rows to delta")


print("Writing delta tables (one parquet + one Spark job per table)...")
write_table(ws_df, "ScanResults")
write_table(dfl_df, "Dataflows")
write_table(ds_df, "Datasets")

# cleanup staging
for sub in ("ScanResults", "Dataflows", "Datasets"):
    shutil.rmtree(f"{STAGING_BASE}/{sub}", ignore_errors=True)

print(f"DONE: {len(ws_df)} workspace rows, {len(dfl_df)} dataflow rows, {len(ds_df)} impacted dataset rows")
